# Example: Dynamical decoupling

The IQM Circuit to Pulse Compiler has a standard compilation stage for automatically applying dynamical decoupling (DD) sequences to an instruction schedule. It is controlled by `CircuitCompilationOptions.dd_mode` for basic circuit execution jobs using `IQMClient`, and the DD compilation stage settings when using the compiler directly. By default the DD stage is disabled.

The `iqm.cpc.compiler.dd` module contains various utility functions the DD stage uses, e.g. for locating specific instructions in a schedule, constructing DD sequences, and replacing select instructions with them. For example, we can find locations of `Wait` instructions on qubit drive channels with a certain minimum duration, and replace them with an arbitrary sequence of `Wait` and `IQPulse` instructions that dynamically decouples the qubit, given that the total duration of the sequence is the same as the duration of the original `Wait`.

This notebook is in two parts:

* **Section A - Simple dynamical decoupling** introduces the `dynamical_decoupling` stage on a single idling qubit: how to enable it, how to define a DD sequence through the settings tree or through the compilation context, and what the resulting schedule and measured results look like with and without DD.
* **Section B - Generalized dynamical decoupling** builds on that and covers the generalized DD implementations: applying one DD strategy to selected qubits, giving each qubit its own strategy, and choosing a strategy per idle window - including the two-part compilation used when the idle windows are not known in advance.

## Setup

In [ ]:
import numpy as np

from IPython.core.display import HTML
from qiskit import QuantumCircuit
from qiskit.compiler import transpile
from qiskit.result import marginal_distribution

from iqm.station_control.interface.models import  MoveGateFrameTrackingMode

from iqm.pulse.playlist.visualisation.base import inspect_playlist
from iqm.iqm_client.util import print_env_vars
from iqm.qiskit_iqm import IQMProvider
from iqm.qiskit_iqm.pulse_level_utils import qiskit_to_iqm, sweep_job_to_qiskit

from iqm.cpc.compiler.dd import (
    IdleDDMap,
    IdleWindow,
    identify_idle_windows_for_settings,
    display_idle_windows,
)

In [ ]:
print_env_vars()

provider = IQMProvider()
backend = provider.get_backend()
iqm_client = backend.client
standard_compiler = iqm_client.get_standard_compiler()
shots = 1000

## Section A: Simple dynamical decoupling

We start from the simplest case: a single qubit left idling while other qubits are kept busy. We compile the same circuit twice, once with the DD stage disabled and once with it enabled, and compare both the schedules and the measured results.

Let's create and transpile a quantum circuit that prepares a superposition on one qubit, and then executes a long sequence of gates on other qubits while the first qubit is idling and experiencing decoherence. Finally we reverse the superposition and measure the qubits.

In [ ]:
qc = QuantumCircuit(3)
for _ in range(1):
    qc.h(0)
    qc.barrier()
for _ in range(100):
    qc.cz(1, 2)
    qc.barrier()
qc.h(0)
qc.measure_all()
qc.draw(output='mpl')

In [ ]:
qc_transpiled = transpile(qc, backend=backend, optimization_level=1)
circuits, compiler = qiskit_to_iqm(backend, qc_transpiled)

Now let us create a pair of Compiler settings objects, one with DD turned on and one without.

In [ ]:
standard_settings = standard_compiler.get_settings(circuits=circuits)
dd_settings = standard_compiler.get_settings(circuits=circuits)

There are two ways to define a DD sequence. First, by changing the attributes of the `dynamical_decoupling` compilation stage. Let's look at the available settings.

* `stages.dynamical_decoupling.apply_dd_strategy.dd_is_disabled` is used to turn the entire compilation stage on and off.
* The default DD strategy to be used if the stage is on is defined in the `stages.dynamical_decoupling.apply_dd_strategy.DDStrategy_*` settings.

We can see that DD is by default disabled, let's turn it on in `dd_settings`.

In [ ]:
standard_settings.stages.dynamical_decoupling.apply_dd_strategy

In [ ]:
dd_settings.stages.dynamical_decoupling.apply_dd_strategy.dd_is_disabled = False

Compiling the circuit using the non-DD settings produces a schedule with a long wait on the drive channel of the idling qubit.

In [ ]:
standard_settings.set_shots(shots)
job_def, context = standard_compiler.compile(circuits=circuits, settings=standard_settings)

HTML(inspect_playlist(job_def.sweep_definition.playlist, [0]))

We then execute this schedule and compute the probability of finding the idling qubit in the |0> state. Ideally this should be one, but due to the decoherence experienced by the superposition state it will be less.

In [ ]:
job_tracker = iqm_client.submit_playlist(job_def, context=context)
job_tracker.wait_for_completion()
qiskit_result = sweep_job_to_qiskit(job_tracker, shots=shots)

counts_orig = qiskit_result.get_counts()
print(f"\nQiskit result counts:\n{counts_orig}\n")

prob = marginal_distribution(counts_orig, indices=[0])['0'] / shots
print(f"Probability of finding qubit 0 it the |0> state (original circuit): {prob}")

Compiling the circuit using the DD settings instead produces a schedule where the long wait on the idling qubit is replaced by a DD sequence.

In [ ]:
dd_settings.set_shots(shots)
job_def, context = standard_compiler.compile(circuits=circuits, settings=dd_settings)

HTML(inspect_playlist(job_def.sweep_definition.playlist, [0]))

We execute the DD schedule and again compute the probability of finding the idling qubit in the |0> state, and will likely see that the probability has gone up, due to the DD sequence partially decoupling it from various decoherence sources and better preserving the superposition state.

In [ ]:
job_tracker = iqm_client.submit_playlist(job_def, context=context)
job_tracker.wait_for_completion()
qiskit_result = sweep_job_to_qiskit(job_tracker, shots=shots)
counts_dd = qiskit_result.get_counts()
print(f"\nQiskit result counts:\n{counts_dd}\n")

prob = marginal_distribution(counts_dd, indices=[0])['0'] / shots
print(f"Probability of finding qubit 0 it the |0> state (dynamically decoupled circuit): {prob}")

### Using the `DDStrategy` class

DD can also be applied on circuit-level jobs using `CircuitCompilationOptions.dd_mode` and `CircuitCompilationOptions.dd_strategy` together with `IQMClient.submit_circuits`, in which case DD will be applied on the serverside.
The `dd_strategy` option can be left unspecified in which case the default strategy is used, or you can provide a custom `DDStrategy`.

When using the Compiler, you can also provide a `DDStrategy` object in the compilation context, which will override the corresponding settings.

In [ ]:
# First, you construct the DDStrategy object
from iqm.station_control.interface.models import DDStrategy

dd_strategy = DDStrategy(
    gate_sequences=[(3, "YXYX", "center")],
)
dd_settings.stages.dynamical_decoupling.apply_dd_strategy.dd_is_disabled = False

In [ ]:
# Then you provide it as the initial context to the Compiler
dd_settings.set_shots(shots)
job_def, context = standard_compiler.compile(
    circuits=circuits,
    settings=dd_settings,
    context={"DDStrategy": dd_strategy}
)

HTML(inspect_playlist(job_def.sweep_definition.playlist, [0]))

In [ ]:
job_tracker = iqm_client.submit_playlist(job_def, context=context)
job_tracker.wait_for_completion()
qiskit_result = sweep_job_to_qiskit(job_tracker, shots=shots)
counts_dd = qiskit_result.get_counts()
print(f"\nQiskit result counts:\n{counts_dd}\n")

prob = marginal_distribution(counts_dd, indices=[0])['0'] / shots
print(f"Probability of finding qubit 0 it the |0> state (dynamically decoupled circuit): {prob}")

## Section B: Generalized dynamical decoupling

This section demonstrates more involved ways of applying DD.
Section A already covered standard DD: how `dd_is_disabled=False` enables the built-in
`STANDARD_DD_STRATEGY`, leaving it `True` disables DD entirely, and how to use `DDStrategy`
to specify the default DD sequences to apply.

Two further workflows are shown here, both reachable through the same standard settings tree:

1. **Qubit-specific DD** - each qubit can have its own DD strategy.
2. **Idling-specific DD** - a specific DD strategy can be chosen for each idle window.
   When the idle window keys are not known in advance, the compilation
   can be run in two parts: first the stages up to the DD stage, then, after inspecting the idle
   windows and choosing the map, the remaining stages.
   Splitting the compilation uses the `Compiler.run_stages` and `Compiler.finalize` methods.

In [ ]:
QB1 = backend.index_to_qubit_name(0)
QB2 = backend.index_to_qubit_name(1)
print(f"Using qubits {QB1}, {QB2}")

### Qubit-specific DD

Apply `XX` (CPMG) on one physical qubit and `YXYX` (XY4) on another in the same
compilation call, through the settings tree. Qubits absent from the map receive the default
DD strategy; set `DDStrategy_gate_sequences_ratio = []` to turn that off.

The circuits below are a delay sweep on two qubits, so both of them idle and both can be
given their own sequence.

In [ ]:
delay_arr = np.linspace(500, 10_000, 21)

def delay_exp_2q(wait_ns: float) -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.h(0); qc.h(1); qc.barrier()
    qc.delay(wait_ns, 0, unit="ns"); qc.delay(wait_ns, 1, unit="ns"); qc.barrier()
    qc.h(0); qc.h(1); qc.measure_all()
    return transpile(qc, backend=backend)

qc_list = [delay_exp_2q(t) for t in delay_arr]
circuits, _ = qiskit_to_iqm(backend, qc_list)
qc_list[0].draw(output='mpl')

In [ ]:
qubit_dd_map = {
    QB1: [
        (5, "YXYX", "center"),
    ],
    QB2: [
        (2, "XX", "asap"),
    ],
}

qb_settings = standard_compiler.get_settings(circuits=circuits)
dd = qb_settings.stages.dynamical_decoupling.apply_dd_strategy
dd.dd_is_disabled = False
dd.qubit_specific_dd = qubit_dd_map
dd

In [ ]:
qb_settings.set_shots(shots)
job_def, context = standard_compiler.compile(circuits=circuits, settings=qb_settings)

# Inspect the first circuit's schedule — QB1 gets centered XY4, QB2 gets left-aligned XX.
HTML(inspect_playlist(job_def.sweep_definition.playlist, [0]))

#### Qubits not in `qubit_specific_dd`

Qubits that are not listed in `qubit_specific_dd` receive the default DD strategy, which is
`STANDARD_DD_STRATEGY` unless you change the `DDStrategy_gate_sequences_*` settings.

To give unlisted qubits **no** DD, set any one of the three `DDStrategy_gate_sequences_*`
settings to an empty list, for example:

```python
dd = settings_qb.stages.dynamical_decoupling.apply_dd_strategy
dd.DDStrategy_gate_sequences_ratio = []  # or ..._gate_pattern_xy = [], or ..._align = []
```

To skip DD on a single listed qubit instead, map it to an empty list (or `None`) in
`qubit_specific_dd`.

### Idling-specific DD

For a circuit with multiple idle windows per qubit, assign a specific DD
sequence to each window, or opt out on selected windows.

#### Inspect, then decide

When the idle-window keys are not known in advance, you can run the compilation in two parts
using `Compiler.run_stages`.

Build a circuit with multiple idles per qubit:

In [ ]:
SHORT_WAIT = 2_00   # ns
LONG_WAIT  = 8_00   # ns

def multi_idle_circuit() -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.h(0); qc.h(1); qc.barrier()
    qc.delay(SHORT_WAIT, 0, unit="ns"); qc.delay(SHORT_WAIT, 1, unit="ns"); qc.barrier()
    qc.cx(0, 1); qc.barrier()
    qc.delay(LONG_WAIT, 0, unit="ns"); qc.delay(LONG_WAIT, 1, unit="ns"); qc.barrier()
    qc.h(0); qc.h(1); qc.measure_all()
    return transpile(qc, backend=backend)

qc = multi_idle_circuit()
circuits, _ = qiskit_to_iqm(backend, [qc])
qc.draw(output='mpl')

Now, run the compilation stages up to the DD stage and inspect the idle windows in the instruction schedule.

In [ ]:
idle_settings = standard_compiler.get_settings(circuits=circuits)
idle_settings.stages.dynamical_decoupling.apply_dd_strategy.dd_is_disabled = False

stages = standard_compiler.circuit_stages + standard_compiler.pulse_stages
dd_idx = [stage.name for stage in stages].index("dynamical_decoupling")
context_idle = standard_compiler.compiler_context(components=None, settings=idle_settings)

# Part 1: run all stages before the DD stage
schedules, context = standard_compiler.run_stages(circuits, context_idle, stages=stages[:dd_idx])

idle_windows = identify_idle_windows_for_settings(schedules, context)
display_idle_windows(idle_windows)

Design chosen after reading the window table:

- first physical qubit, idle 0 (SHORT_WAIT): **`XX`** (CPMG), center-aligned
- first physical qubit, idle 1 (LONG_WAIT): **`YXYX`** (XY4), asap-aligned
- second physical qubit, idle 0 (SHORT_WAIT): **`None`** — skip DD entirely
- second physical qubit, idle 1 (LONG_WAIT): **`XX`** (CPMG), alap-aligned

Any idle *not* listed uses the sequences given in the `DDStrategy_gate_sequences_*` settings,
or gets no DD if those are empty.

Use the keys from `idle_windows` to build `idle_dd_map` in the settings, and run the remaining compilation stages.

In [ ]:
idle_qubits = []
for window in idle_windows:
    if window.schedule_index == 0 and window.qubit not in idle_qubits:
        idle_qubits.append(window.qubit)

if len(idle_qubits) < 2:
    raise RuntimeError(f"Expected idle windows on two qubits, found {idle_qubits!r}")

idle_qubit_a, idle_qubit_b = idle_qubits[:2]

idle_dd_map: IdleDDMap = {
    (0, idle_qubit_a, 0): [(2, "XX",   "center")],
    (0, idle_qubit_a, 1): [(5, "YXYX", "asap")],
    (0, idle_qubit_b, 0): [],
    (0, idle_qubit_b, 1): [(2, "XX",   "alap")],
}

available_idle_keys = {window.key for window in idle_windows}
unknown_idle_keys = set(idle_dd_map) - available_idle_keys
if unknown_idle_keys:
    raise ValueError(f"IdleDDMap contains keys that are not in idle_windows: {sorted(unknown_idle_keys)}")

idle_settings.stages.dynamical_decoupling.apply_dd_strategy.idling_specific_dd = idle_dd_map
idle_settings.set_shots(shots)

# Part 2: run the DD stage and all stages after it, then finalize
schedules, context = standard_compiler.run_stages(schedules, context, stages=stages[dd_idx:])
job_def, context = standard_compiler.finalize(schedules, context)

# First discovered qubit: idle 0 -> CPMG, idle 1 -> XY4.
# Second discovered qubit: idle 0 -> plain Wait, idle 1 -> CPMG.
HTML(inspect_playlist(job_def.sweep_definition.playlist, [0]))

#### Idling-specific DD when the `IdleDDMap` is already known

If the idle-window keys are already known, no inspection step is needed. This mirrors the
qubit-specific DD settings workflow: create the map, assign it to
`stages.dynamical_decoupling.apply_dd_strategy.idling_specific_dd`, and call
`standard_compiler.compile(...)` once.

`IdleDDMap` keys are `(schedule_index, physical_qubit, idle_index)`. Values are either a
DD gate-sequence list or `None` to skip DD on that idle.

In [ ]:
# In a real known-map workflow these keys come from prior calibration, prior inspection,
# or documentation for the exact compiled circuit family. In this notebook we reuse
# the physical qubits discovered above so the demo remains valid after routing to physical qubits.
known_idle_dd_map: IdleDDMap = {
    (0, idle_qubit_a, 0): [(2, "XX",   "center")],
    (0, idle_qubit_a, 1): [(5, "YXYX", "asap")],
    (0, idle_qubit_b, 0): [],
    (0, idle_qubit_b, 1): [(2, "XX",   "alap")],
}

unknown_known_keys = set(known_idle_dd_map) - available_idle_keys
if unknown_known_keys:
    raise ValueError(f"known_idle_dd_map contains keys not present in idle_windows: {sorted(unknown_known_keys)}")

known_idle_dd_map

In [ ]:
idle_settings = standard_compiler.get_settings(circuits=circuits)
dd = idle_settings.stages.dynamical_decoupling.apply_dd_strategy
dd.dd_is_disabled = False
dd.idling_specific_dd = known_idle_dd_map

job_def, context = standard_compiler.compile(
    circuits=circuits,
    settings=idle_settings,
)
HTML(inspect_playlist(job_def.sweep_definition.playlist, [0]))

#### Idle windows not in `idling_specific_dd`

Idle windows that are not listed in `idling_specific_dd` receive the default DD strategy, which is
`STANDARD_DD_STRATEGY` unless you change the `DDStrategy_gate_sequences_*` settings.

To give unlisted idle windows **no** DD, set any one of the three `DDStrategy_gate_sequences_*`
settings to an empty list, for example:

```python
dd.DDStrategy_gate_sequences_ratio = []  # or ..._gate_pattern_xy = [], or ..._align = []
```

To skip DD on a single listed idle window instead, map its key to an empty list (or `None`) in
`idling_specific_dd`.